# Step 6 - Building Feature Batches (batch_XXX.npz)

This step assembles encoded videos into `.npz` files that match the author's format and
composition:

- `data`: `(10, 100, 20, 4096)`, float16
- `labels`: `(10, 2)`, float16, one-hot `[no-accident, accident]`

Each batch contains **5 accident and 5 non-accident** videos in the fixed order
`[n, n, n, A, A, A, n, n, A, A]`, which matches every batch in the author's dataset. Because
the dataset is imbalanced (610 accident versus 1130 non-accident videos), the accident pool is
oversampled (cycled across batches) so that every batch remains balanced.

Output is written to `Dataset/features_generated/`, leaving the author's reference files
unchanged.

## Models, helpers, and per-video encoder

Consolidated from Steps 2-5.

In [1]:
import cv2, time, glob, os
import numpy as np
import torch
import torch.nn as nn

N_FRAMES=100; N_DETECTION=20; N_OBJECTS=19; FEAT_DIM=4096
KEEP={1,2,3,4,6,7,8}; SCORE_THRESH=0.5
IMAGENET_MEAN=torch.tensor([0.485,0.456,0.406]).view(1,3,1,1)
IMAGENET_STD =torch.tensor([0.229,0.224,0.225]).view(1,3,1,1)

from torchvision.models.detection import fasterrcnn_resnet50_fpn
try:
    from torchvision.models.detection import FasterRCNN_ResNet50_FPN_Weights
    detector = fasterrcnn_resnet50_fpn(weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT)
except Exception:
    detector = fasterrcnn_resnet50_fpn(pretrained=True)
detector.eval()

from torchvision.models import vgg16
try:
    from torchvision.models import VGG16_Weights
    vgg = vgg16(weights=VGG16_Weights.IMAGENET1K_V1)
except Exception:
    vgg = vgg16(pretrained=True)
vgg.eval()
vgg_features=vgg.features; vgg_avgpool=vgg.avgpool
vgg_fc7=nn.Sequential(*list(vgg.classifier.children())[:5])

def detect_boxes(frame):
    t=torch.from_numpy(frame).permute(2,0,1).float()/255.0
    with torch.no_grad():
        out=detector([t])[0]
    kept=[(float(s),b) for b,l,s in zip(out['boxes'].numpy(),out['labels'].numpy(),out['scores'].numpy())
          if s>=SCORE_THRESH and int(l) in KEEP]
    kept.sort(key=lambda z:z[0], reverse=True)
    return [b for _,b in kept[:N_OBJECTS]]

def crop_box(frame, box):
    h,w,_=frame.shape; x1,y1,x2,y2=box
    x1=max(0,int(round(x1))); y1=max(0,int(round(y1)))
    x2=min(w,int(round(x2))); y2=min(h,int(round(y2)))
    return None if (x2-x1<2 or y2-y1<2) else frame[y1:y2,x1:x2]

def to_features_batch(crops):
    if not crops: return np.zeros((0,FEAT_DIM),np.float32)
    batch=np.stack([cv2.resize(c,(224,224)) for c in crops]).astype(np.float32)/255.0
    x=torch.from_numpy(batch).permute(0,3,1,2)
    x=(x-IMAGENET_MEAN)/IMAGENET_STD
    with torch.no_grad():
        x=vgg_features(x); x=vgg_avgpool(x); x=torch.flatten(x,1); x=vgg_fc7(x)
    return x.numpy().astype(np.float32)

def encode_video(path, max_frames=100):
    cap=cv2.VideoCapture(path); frames=[]
    while True:
        ok,f=cap.read()
        if not ok: break
        frames.append(cv2.cvtColor(f,cv2.COLOR_BGR2RGB))
    cap.release()
    out=np.zeros((N_FRAMES,N_DETECTION,FEAT_DIM),np.float32)
    for t in range(min(max_frames,len(frames))):
        frame=frames[t]; crops=[frame]
        for box in detect_boxes(frame):
            c=crop_box(frame,box)
            if c is not None: crops.append(c)
            if len(crops)==N_DETECTION: break
        feats=to_features_batch(crops)
        out[t,:feats.shape[0],:]=feats
    return out
print('ready.')

ready.


## Batch composition

Each batch draws 5 accident and 5 non-accident videos and arranges them in the fixed order.
`plan_batch` uses modulo indexing so the accident pool cycles, implementing the oversampling.

In [3]:
# ----------------- CONFIG -----------------
SPLIT      = 'train'    # 'train' or 'test'
N_BATCHES  = 1         # how many batch_XXX.npz to build
MAX_FRAMES = 5        # 20 = quick test, 100 = full
# ------------------------------------------

ORDER = ['n','n','n','A','A','A','n','n','A','A']   # author's fixed per-batch order
LABEL = {'A':(0.0,1.0), 'n':(1.0,0.0)}

acc_pool = sorted(glob.glob(f'../Dataset/videos/{SPLIT}/Accident/*.mp4'))
non_pool = sorted(glob.glob(f'../Dataset/videos/{SPLIT}/NonAccident/*.mp4'))
print(f'{SPLIT}: {len(acc_pool)} accident videos, {len(non_pool)} non-accident videos')
assert acc_pool and non_pool, 'no videos found - check SPLIT / paths'

def plan_batch(b):
    """Return the ordered list of (video_path, label) for batch index b (0-based).
    Uses modulo indexing so the accident pool cycles (oversampling)."""
    acc = [acc_pool[(b*5 + k) % len(acc_pool)] for k in range(5)]
    non = [non_pool[(b*5 + k) % len(non_pool)] for k in range(5)]
    items, ai, ni = [], 0, 0
    for slot in ORDER:
        if slot == 'A':
            items.append((acc[ai], LABEL['A'])); ai += 1
        else:
            items.append((non[ni], LABEL['n'])); ni += 1
    return items

# show the plan for the first batch
print('\nbatch_001 plan:')
for p, lab in plan_batch(0):
    print('  ', 'A' if lab[1]==1 else 'n', os.path.basename(p))

train: 445 accident videos, 829 non-accident videos

batch_001 plan:
   n 000001.mp4
   n 000002.mp4
   n 000003.mp4
   A 000011.mp4
   A 000012.mp4
   A 000013.mp4
   n 000004.mp4
   n 000005.mp4
   A 000014.mp4
   A 000015.mp4


## Build and save the batches

In [4]:
out_dir = f'../Dataset/features_generated/{SPLIT}'
os.makedirs(out_dir, exist_ok=True)

for b in range(N_BATCHES):
    items = plan_batch(b)
    data   = np.zeros((10, N_FRAMES, N_DETECTION, FEAT_DIM), np.float32)
    labels = np.zeros((10, 2), np.float32)
    t0 = time.time()
    print(f'=== batch_{b+1:03d} ===')
    for i,(path,lab) in enumerate(items):
        ti=time.time()
        data[i]  = encode_video(path, MAX_FRAMES)
        labels[i]= lab
        print(f'  [{i+1:2d}/10] {"A" if lab[1]==1 else "n"} {os.path.basename(path):12s} {time.time()-ti:5.1f}s')
    out_path = f'{out_dir}/batch_{b+1:03d}.npz'
    np.savez(out_path, data=data.astype(np.float16), labels=labels.astype(np.float16))
    print(f'  saved {out_path}  ({os.path.getsize(out_path)/1e6:.1f} MB)  total {time.time()-t0:.1f}s\n')

=== batch_001 ===
  [ 1/10] n 000001.mp4    41.2s
  [ 2/10] n 000002.mp4    38.7s
  [ 3/10] n 000003.mp4    42.9s
  [ 4/10] A 000011.mp4    45.2s
  [ 5/10] A 000012.mp4    46.0s
  [ 6/10] A 000013.mp4    52.7s
  [ 7/10] n 000004.mp4    41.2s
  [ 8/10] n 000005.mp4    42.6s
  [ 9/10] A 000014.mp4    25.8s
  [10/10] A 000015.mp4    39.8s
  saved ../Dataset/features_generated/train/batch_001.npz  (163.8 MB)  total 417.0s



## Verify a saved batch

Confirms the shape, dtype, class composition, and ordering against the author's pattern.

In [5]:
f = np.load(f'{out_dir}/batch_001.npz')
data, labels = f['data'], f['labels']
n_acc = int(sum(1 for r in labels if r[1]==1))
print('data  :', data.shape, data.dtype)
print('labels:', labels.shape, labels.dtype)
print('composition:', n_acc, 'accident +', 10-n_acc, 'non-accident')
print('order :', ['A' if r[1]==1 else 'n' for r in labels])
print('data >= 0 ?', bool((data>=0).all()), ' max:', round(float(data.max()),2))
print('\nmatches author pattern [n,n,n,A,A,A,n,n,A,A]?',
      ['A' if r[1]==1 else 'n' for r in labels] == ['n','n','n','A','A','A','n','n','A','A'])

data  : (10, 100, 20, 4096) float16
labels: (10, 2) float16
composition: 5 accident + 5 non-accident
order : ['n', 'n', 'n', 'A', 'A', 'A', 'n', 'n', 'A', 'A']
data >= 0 ? True  max: 6.59

matches author pattern [n,n,n,A,A,A,n,n,A,A]? True


## Summary

- Each batch is 5 accident + 5 non-accident videos in the author's fixed order.
- Accident videos are oversampled by cycling the pool, which balances the imbalanced dataset.
- Set `MAX_FRAMES = 100` and increase `N_BATCHES` for full runs.